In [0]:

import os, sys
try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    if "/repoint/" in notebook_path:
        repo_root = "/Workspace" + notebook_path.rsplit("/repoint/", 1)[0]
    else:
        repo_root = "/Workspace/Users/extkgentekaki@cepal.gr/gr.cepal.dna.de.delivery"
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname

env = get_env_prefix()
env_name = get_env_fullname()

SRC_PAYMENT_LEAF   = f"{env_name}.landed.qca_fs_payment_leaf"
SRC_CASES_STATIC   = f"{env_name}.landed.qca_dim_cases_static"
SRC_CUSTOMER_STATIC = f"{env_name}.landed.qca_dim_customer_static"
SRC_BUCKET         = f"{env_name}.landed.qca_dim_bucket"

# ---- Resolve run_date ----
dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    # run_date empty: fall back to max business date
    run_date = str(
        spark.table(SRC_PAYMENT_LEAF)
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

# pay_month = YYYYMM of the PREVIOUS month's last WORKING day, from ref_calendar
# (holiday/weekend aware, year-boundary safe). CalendarDate is a string column.
_cal = spark.sql(f'''
    SELECT PreviousMonthLastWorkingDate AS LastWorkingDayPrevMonth
    FROM {env_name}.reference.ref_calendar
    WHERE CalendarDate = '{run_date}'
''').collect()
if not _cal or _cal[0]['LastWorkingDayPrevMonth'] is None:
    raise ValueError(f"No PreviousMonthLastWorkingDate found in ref_calendar for run_date={run_date}")
pay_month = str(_cal[0]['LastWorkingDayPrevMonth'])[:7].replace('-', '')   # e.g. '202511'

target_area = f"`{env}-cdh-delivery`.repoint"
target_table = f"{target_area}.nfpayments"
partition_columns = ["BusinessDate"]

print(f"run_date: {run_date} | pay_month: {pay_month} | target: {target_table}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DecimalType, DateType

_pl_all = spark.table(SRC_PAYMENT_LEAF).filter(F.col("BusinessDate") <= F.lit(run_date).cast("date"))
_pl_bd = _pl_all.agg(F.max("BusinessDate").alias("bd")).collect()[0]["bd"]
if _pl_bd is None:
    raise ValueError(f"No {SRC_PAYMENT_LEAF} rows on/before run_date={run_date}")

payment_leaf = (
    _pl_all.filter(F.col("BusinessDate") == F.lit(_pl_bd))
    .filter(F.date_format(F.col("PD_DESC").cast("date"), "yyyyMM") == F.lit(pay_month))
)

def _latest_dim(table):
    src = spark.table(table).filter(F.col("BusinessDate") <= F.lit(_pl_bd))
    bd = src.agg(F.max("BusinessDate").alias("bd")).collect()[0]["bd"]
    if bd is None:
        raise ValueError(f"No {table} rows on/before {_pl_bd}")
    return src.filter(F.col("BusinessDate") == F.lit(bd))

dim_cases_static    = _latest_dim(SRC_CASES_STATIC)
dim_customer_static = _latest_dim(SRC_CUSTOMER_STATIC)
dim_bucket          = _latest_dim(SRC_BUCKET)

_pay_desc_data = [
    ("074--100", "ALPHABANK GR N/A"),
    ("074-074ABH-0010", "NEW_ΣΠΥΡΑΝΤΗ ΑΓΑΠΗ"),
    ("074-074C01-101", "PAYMENT MANUAL"),
    ("074-074C01-113", "PAYMENT AUTOMATIC"),
    ("074-074C01-133", "PAYMENT ATM"),
    ("074-074C01-171", "PAYMENT"),
    ("074-074C02-101", "PAYMENT MANUAL"),
    ("074-074C02-113", "PAYMENT AUTOMATIC"),
    ("074-074C02-133", "PAYMENT ATM"),
    ("074-074C02-171", "PAYMENT"),
    ("074-074C02-233", "NEW_MC MONEYSEND PAYMENT LOCAL"),
    ("074-074C02-235", "NEW_MC MONEYSEND PAYMENT INAL"),
    ("074-074C02-244", "NEW_PAYMENT AUCTION"),
    ("074-074C02-653", "NEW_MIGR CTD PAYMENTS"),
    ("074-074C03-101", "PAYMENT MANUAL"),
    ("074-074C03-113", "PAYMENT AUTOMATIC"),
    ("074-074C03-133", "PAYMENT ATM"),
    ("074-074C03-171", "PAYMENT"),
    ("074-074C18-101", "PAYMENT MANUAL"),
    ("074-074C18-113", "PAYMENT AUTOMATIC"),
    ("074-074C18-133", "PAYMENT ATM"),
    ("074-074C18-171", "PAYMENT"),
    ("074-074C18-265", "TRANSFER ALL IN ONE"),
    ("074-074C18-267", "TRANSFER ARRANGEMENT"),
    ("074-074C23-101", "PAYMENT MANUAL"),
    ("074-074C25-101", "PAYMENT MANUAL"),
    ("074-074C25-113", "PAYMENT AUTOMATIC"),
    ("074-074C25-133", "PAYMENT ATM"),
    ("074-074C25-171", "PAYMENT"),
    ("074-074C25-244", "NEW_PAYMENT AUCTION"),
    ("074-074C36-101", "PAYMENT MANUAL"),
    ("074-074C36-113", "PAYMENT AUTOMATIC"),
    ("074-074C36-133", "PAYMENT ATM"),
    ("074-074C36-171", "PAYMENT"),
    ("074-074C36-233", "NEW_MC MONEYSEND PAYMENT LOCAL"),
    ("074-074C36-235", "NEW_MC MONEYSEND PAYMENT INAL"),
    ("074-074C36-653", "NEW_MIGR CTD PAYMENTS"),
    ("074-074C37-101", "PAYMENT MANUAL"),
    ("074-074C37-113", "PAYMENT AUTOMATIC"),
    ("074-074C37-133", "PAYMENT ATM"),
    ("074-074C37-171", "PAYMENT"),
    ("074-074PRF-4211", "NEW_ΕΙΣΑΓΩΓΗ ΕΞΟΦΛΗΣΗΣ - ΑΠΑΙΤΗΣΗΣ"),
    ("074-074PRF-4221", "NEW_ΕΙΣΑΓΩΓΗ ΕΙΣΠΡΑΞΗΣ ΑΠΑΙΤΗΣΗΣ"),
    ("074-074PRF-4231", "NEW_ΕΙΣΑΓΩΓΗ ΠΡΟΩΡΗΣ ΟΛΙΚΗΣ ΕΞΟΦΛΗΣΗΣ ΤΟΚ/ΗΣ"),
    ("074-074PRF-4271", "NEW_ΕΙΣΑΓΩΓΗ ΚΛΕΙΣΙΜΟ ΛΟΓΑΡΙΑΣΜΟΥ - ΠΡΟΒΛΕΨΗ"),
    ("074-074PRF-4271                ", "NEW_ΕΙΣΑΓΩΓΗ ΚΛΕΙΣΙΜΟ ΛΟΓΑΡΙΑΣΜΟΥ - ΠΡΟΒΛΕΨΗ"),
    ("074-074PRF-4291", "NEW_ΕΙΣΑΓΩΓΗ ΠΡΟΩΡΗΣ ΟΛΙΚΗΣ ΕΞ ΤΟΚ/ΗΣ ΧΡ/ΚΟΥ"),
    ("074-074PRF-4291                ", "NEW_ΕΙΣΑΓΩΓΗ ΠΡΟΩΡΗΣ ΟΛΙΚΗΣ ΕΞ ΤΟΚ/ΗΣ ΧΡ/ΚΟΥ"),
    ("074-074PRF-4301", "NEW_ΕΙΣΑΓΩΓΗ ΠΡΟΩΡΗ ΜΕΡΙΚΗ ΕΞΟΦΛΗΣΗΣ ΔΑΝΕΙΟΥ"),
    ("074-074PRF-4311                ", "NEW_ΕΙΣΑΓΩΓΗ ΑΚΥΡΩΣΗΣ                       "),
    ("074-074PRF-4331", "NEW_ΕΙΣΑΓΩΓΗ ΠΡΟΩΡΗΣ ΜΕΡΙΚΗΣ ΕΙΣ. ΕΠΟΜ. ΔΟΣ"),
    ("074-074PRF-4331                ", "NEW_ΕΙΣΑΓΩΓΗ ΠΡΟΩΡΗΣ ΜΕΡΙΚΗΣ ΕΙΣ. ΕΠΟΜ. ΔΟΣ "),
    ("074-074PRF-4981", "NEW_ΕΙΣΠΡΑΞΗ ΕΝΑΝΤΙ ΛΟΓΑΡΙΑΣΜΟΥ"),
    ("074-074PRF-4981                ", "NEW_ΕΙΣΠΡΑΞΗ ΕΝΑΝΤΙ ΛΟΓΑΡΙΑΣΜΟΥ             "),
    ("074-074PRF-9576", "NEW_CHF - ΕΙΣΠΡΑΞΗ ΕΝΑΝΤΙ ΛΟΓΑΡΙΑΣΜΟΥ"),
    ("074-074PRF-Cash", "074PRF-Cash"),
    ("074-074SGL-30822", "ΠΡΟΩΡΗ ΕΞΟΦ. ΔΑΝΕΙΟΥ"),
    ("074-074SGL-30852", "ΕΞΟΦΛΗΣΗ ΔΑΝΕΙΟΥ"),
    ("074-074SGL-50806", "ΠΡΟΩΡ.ΕΞΟΦ.ΚΕΦΑΛ"),
    ("074-074SGL-50814", "ΜΕΡ.ΑΠΟΠΛΗΡ.ΛΟΓΑΡ."),
    ("074-074SGL-50815", "ΕΙΣΠΡ.ΚΑΘ/ΝΟΥ ΜΕΡ."),
    ("074-074SGL-50816", "ΟΛΙΚΗ ΑΠΟΠΛΗΡ. ΛΟΓΑΡ"),
    ("074-074SGL-50870", "ΚΑΤ.ΕΝΑΝΤΙ ΔΟΣΗΣ/ΤΟΚ"),
    ("074-074SGL-50871", "ΠΙΣΤ. ΕΝΑΝΤΙ. ΛΟΓ."),
    ("074-074SGL-51260", "ΠΛΗΡ. ΑΠΟΣΒ. ΑΠΑΙΤΗΣ"),
    ("074-074SGL-61206", "ΠΛΗΡ ΛΟΓ ΣΕ ΔΙΚ"),
]
_pay_desc_map = spark.createDataFrame(_pay_desc_data, ["PAY_TR_CODE", "PAY_DESCRIPTION"])

print(f"payment_leaf snapshot BusinessDate: {_pl_bd}")

In [0]:
pl = payment_leaf.select(
    F.col("CASE_ID").alias("PL_CASE_ID"),
    F.col("BCK_BC_ID").alias("PL_BCK_BC_ID"),
    F.col("PAYR_ID").alias("PL_PAYR_ID"),
    F.col("BU_ID").alias("BUSINNES_UNIT"),
    F.col("PAY_CODE"),
    F.col("PD_DESC").alias("PAY_DATE"),
    F.col("PAY_POST_DATE"),
    F.col("PAY_CREATED"),
    F.col("FSPM_AMT_PMT"),
    F.col("FSPM_AMT_OUT").alias("OUTSTANDING_BALANCE"),
    F.col("FSPM_AMT_DBT").alias("DEBT_AMOUNT"),
    F.col("PAY_CURRENCY").alias("CURRENCY"),
    F.col("MULTIPLY_FOR_LCY"),
    F.col("PAY_GENERIC_TEXT_01").alias("PAYMENT_REASON_DESCRIPTION"),
    F.col("PAY_GENERIC_TEXT_02").alias("PAYMENT_RESON_CODE"),
    F.col("DENO_ID").alias("DENOUNCE_STATUS"),
    F.col("PAYR_ID").alias("PAY_TR_CODE"),      
    F.substring(F.col("BU_ID"), 5, 2).alias("PL_BU_CODE"),
)

cs = dim_cases_static.select(
    F.col("CASE_CODE").alias("CS_CASE_CODE"),
    F.col("CUST_CODE").alias("CS_CUST_CODE"),
    F.col("CASE_CONTR_NUM").alias("ACCOUNT_NUMBER"),
    F.col("CASE_LEGAL_CONTRACT").alias("CONTRACT_NUMBER"),
    F.col("CASE_BU").alias("CS_CASE_BU"),
)

cust = dim_customer_static.select(
    F.col("CUST_ID").alias("CU_CUST_ID"),
    F.col("CUST_CIF").alias("CIF"),
    F.col("CUST_LAST_NAME"),
    F.col("CUST_FIRST_NAME"),
    F.col("CUST_FATH_NAME"),
    F.col("CUST_AFM").alias("AFM"),
)

bck = dim_bucket.select(
    F.col("BCK_ID").alias("BK_BCK_ID"),
    F.col("BCK_DESC").alias("PAY_BUCKET"),
)
pay_desc_lkp = _pay_desc_map.alias("pdl")

In [0]:
joined = (
    pl
    .join(F.broadcast(cs),
          (pl["PL_CASE_ID"] == cs["CS_CASE_CODE"]) & (pl["PL_BU_CODE"] == cs["CS_CASE_BU"]),
          "inner")
    .join(F.broadcast(cust), cust["CU_CUST_ID"] == cs["CS_CUST_CODE"], "inner")
    .join(F.broadcast(bck), bck["BK_BCK_ID"] == pl["PL_BCK_BC_ID"], "inner")
    .join(F.broadcast(pay_desc_lkp), F.col("pdl.PAY_TR_CODE") == pl["PAY_TR_CODE"], "left")
    .drop(F.col("pdl.PAY_TR_CODE"))  
)

In [0]:
pay_amt_lcy_num = (F.col("FSPM_AMT_PMT") * F.col("MULTIPLY_FOR_LCY")).cast(DecimalType(18, 5))
pay_amt_lcy_str = pay_amt_lcy_num.cast(StringType())  

final_df = joined.select(
    F.col("PAY_CODE").cast(StringType()).alias("PAY_CODE"),
    F.col("ACCOUNT_NUMBER").cast(StringType()).alias("ACCOUNT_NUMBER"),
    F.col("CONTRACT_NUMBER").cast(StringType()).alias("CONTRACT_NUMBER"),
    F.col("CIF").cast(StringType()).alias("CIF"),
    F.col("PAY_DATE").cast(DateType()).alias("PAY_DATE"),
    F.col("PAY_POST_DATE").cast(DateType()).alias("PAY_POST_DATE"),
    F.col("PAY_CREATED").cast(DateType()).alias("PAY_CREATED"),
    F.regexp_replace(F.col("FSPM_AMT_PMT").cast(StringType()), "\\.0$", "").alias("PAYMENT_AMOUNT"),
    pay_amt_lcy_str.alias("PAY_AMT_LCY"),
    F.col("CURRENCY").cast(StringType()).alias("CURRENCY"),
    F.regexp_replace(F.col("OUTSTANDING_BALANCE").cast(StringType()), "\\.0$", "").alias("OUTSTANDING_BALANCE"),
    F.regexp_replace(F.col("DEBT_AMOUNT").cast(StringType()), "\\.0$", "").alias("DEBT_AMOUNT"),
    F.col("PAY_BUCKET").cast(StringType()).alias("PAY_BUCKET"),
    F.col("BUSINNES_UNIT").cast(StringType()).alias("BUSINNES_UNIT"),
    F.col("PAY_TR_CODE").cast(StringType()).alias("PAY_TR_CODE"),
    F.col("PAYMENT_RESON_CODE").cast(StringType()).alias("PAYMENT_RESON_CODE"),
    F.col("PAYMENT_REASON_DESCRIPTION").cast(StringType()).alias("PAYMENT_REASON_DESCRIPTION"),
    F.col("CUST_LAST_NAME").cast(StringType()).alias("CUST_LAST_NAME"),
    F.col("CUST_FIRST_NAME").cast(StringType()).alias("CUST_FIRST_NAME"),
    F.col("CUST_FATH_NAME").cast(StringType()).alias("CUST_FATH_NAME"),
    F.col("AFM").cast(StringType()).alias("AFM"),
    F.col("pdl.PAY_DESCRIPTION").cast(StringType()).alias("PAY_DESCRIPTION"),
    F.col("DENOUNCE_STATUS").cast(StringType()).alias("DENOUNCE_STATUS"),
).distinct()

# ORDER BY ACCOUNT_NUMBER (on-prem) — sort is not persisted by Delta but kept for parity/inspection.
final_df = final_df.orderBy(F.col("ACCOUNT_NUMBER"))

In [0]:
from pyspark.sql.functions import current_timestamp, from_utc_timestamp, lit

final_df_write = final_df.withColumn(
    "BusinessDate", lit(run_date).cast(DateType())
).withColumn(
    "IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
)

if spark.catalog.tableExists(target_table):
    spark.sql(f"DELETE FROM {target_table} WHERE BusinessDate = '{run_date}'")

final_df_write.write \
    .format("delta") \
    .partitionBy(*partition_columns) \
    .option("replaceWhere", f"BusinessDate = '{run_date}'") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(target_table)

spark.sql(f"REPAIR TABLE {target_table} SYNC METADATA")

print(f"Written to {target_table} | BusinessDate={run_date} | pay_month={pay_month}")